# Map Coordinates 

## 1. Data Files

The data files to be loaded are:

- Attraction file name: attractions_with_coordinates.csv  


## Data Description

Assume the data files already exist locally.

The datasets have the following structure:

### attractions_with_coordinates.csv 
- Each row represents one attraction.
- Columns:
  - [totalScore] (float): overall rating of the attraction.
  - [latitude] (float): latitude of the attraction.
  - [longitude] (float): longitude of the attraction.



## 2. What I Want to See on the Map

I want the map to show:

- All attraction locations  

For each attraction marker, I want to display:

- The attraction number and its overall rating.



## 3. Final Output

The final output should be:

- An interactive map showing all attractions.


## Response Rules for GenAI

Please strictly follow these rules:

1. Assume all CSV files already exist locally in the working directory.
2. The first action in the code must be loading the three CSV files using pandas.
3. Do NOT check whether the files exist and do NOT ask me to upload them.
4. Do NOT comment on execution environment limitations.
5. Only generate code needed to load the data and display the map.
6. Do NOT perform any optimization, modeling, or analysis.
7. Do NOT introduce additional datasets or advanced features.
8. Use `pandas` and `folium`.
9. Keep the code simple and easy to run.
10. Do NOT add long explanations or non-code text.


## Task for GenAI
Please generate **Python code** that:

- Check and install required packages (`pandas`, `folium`) if necessary
- Imports required libraries (`pandas`, `folium`).
- Loads [the CSV file "attractions_with_coordinates.csv"].
- Creates [a map that shows all attraction locations. For each attraction marker, I want to display the attraction number and its overall rating].
- Displays attraction locations with simple popups.
- Shows the map as the final output.

**Only output the code.**

In [2]:
import sys
import subprocess
import importlib

for package in ["pandas", "folium"]:
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import pandas as pd
import folium

attractions = pd.read_csv("attractions_with_coordinates_price.csv")

center_lat = attractions["latitude"].mean()
center_lon = attractions["longitude"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

for i, row in attractions.reset_index(drop=True).iterrows():
    popup_text = f"Attraction {i+1}<br>Name: {row['title']}<br>Rating: {row['totalScore']}"
    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=folium.Popup(popup_text, max_width=250)
    ).add_to(m)

m

Assume the role of a data engineer specialized in data and prescriptive analytics. You are proficient in constructing mathematical models and adept at programming and solving these models using the Gurobi library. In our subsequent interactions, you will be tasked with coding various optimization models utilizing the Gurobi library. Your objectives are to ensure accurate results and adhere to the following guidelines:

1. Generate random parameters when specific parameters are not provided.
    
2. After solving each optimization problem, verify the problem status. If the status is "OPTIMAL", include code snippets to retrieve and display the optimal solutions and the objective value.

3. Disable output logging by setting the "OutputFlag" to 0.

4. Present the results in a concise and clear format, utilizing tables or graphical representations when beneficial.

5. Strive for code efficiency, utilizing the minimal amount of code necessary for model programming.

6. If possible, use “addVars” for non-scalar decision variables and use “addConstrs” for a batch of constraints. Also, please use “quicksum” instead of “sum”.

7. Be careful of the indices given that Python starts from 0 but m ost of the models start from 1. 

Suppose that I am visiting Tokyo for 3 days and I want to visit up to 3 attractions each day. In total, I will be visiting 9 attractions in total. Here are my criteria:
1. I want to visit at most 2 shopping malls
2. I will either visit an attraction that is a Buddhist temple or a Shinto shrine
3. I do not want to visit a Tourist information center
4. I will only visit at most 3 attractions in the Shinjuku District 
5. I will only go to at least one amusement park/amusement center/theme park
6. If I visit the Tokyo Dome City, then I must visit Shinjuku Gyoen Cherry Tree Area
7. I do want to visit Tokyo Camii & Diyanet Turkish Culture Center
8. I want to visit at most either 2 of these attractions ["Ninja Trick House In Tokyo", "Samurai Ninja Museum Asakusa Tokyo" and "Asakusa Hanayashiki"]

# Plan 1: Adding budget into the constraint 

Assume that I want to create app that helps to tourist to pick attractions to go to by their requirements. They want to go to exactly 9 attractions in Tokyo. Suppose that I have n attractions and m categories. Here are my criterias:

Variables:
- y_i: Binary variable where attraction i is selected (Decision Variable)
- r_i: Review score of attraction i 
- e_i: Entrance fee of each attractions, column name "prices" 
- A : A set of attraction which is a subset of {1,...,n}
- C : A set of categories which is a subset of {1,...,m}
- a_{ij}: Binary variable if attraction i is truely category j 
- z_j: A binary variable where z_j = 1 if category j is represented at least once, otherwise z_j = 0
- g_i: Binary variable where g_i = 1 if attraction i belongs to at least one category in group C

Objective function: sum{i = 1 to n}(r_i * y_i)

s.t
- sum{i = 1 to n} y_i = 9
- sum{i = 1 to n} a_{i, "Shopping mall"} * yi <= 2
- z_{"Buddhist temple"} + z_{"Shinto shrine"} <= 1
- sum{i = 1 to n} a_{i, "Information centre"} * yi <= 0
- sum{i is an element of a set that contains "Shinjuku" under the "city" column in attractions } y_i <= 3
- sum{i = 1 to n} g_i * y_i >=  1 where g_i = 1 if attraction i belongs to at least "Amusement park", "Asumement center" or "Theme park"
- y_{"Tokyo Dome City} <= y_{"Shinjuku Gyoen Cherry Tree Area"}
- y{"Tokyo Camii & Diyanet Turkish Culture Center"} = 0 
- y_{"Ninja Trick House In Tokyo"} + y_{"Samurai Ninja Museum Asakusa Tokyo"} + y_{"Asakusa Hanayashiki"} <=2
- y_i is binary
- sum{i = 1 to n} e_i * y_i <= E, where E is my budget and by default, set it as the average price * number of attractions I intended to go (i.e. 9 attractions here)

Can you provide Python code using Gurobi to solve the problem? Please make the solver a function named "solve_attraction_finding_problem". 

In [16]:
def solve_attraction_finding_problem(attractions, budget=None):
    import pandas as pd
    import numpy as np
    import gurobipy as gp
    from gurobipy import GRB, quicksum

    df = attractions.copy().reset_index(drop=True)

    # -------------------------------------------------
    # Basic validation
    # -------------------------------------------------
    required_cols = ["title", "city", "prices"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required column(s): {missing_cols}")

    # -------------------------------------------------
    # Review score column
    # -------------------------------------------------
    if "review_score" in df.columns:
        score_col = "review_score"
    elif "totalScore" in df.columns:
        score_col = "totalScore"
    elif "r_i" in df.columns:
        score_col = "r_i"
    else:
        raise ValueError("No review score column found.")

    # -------------------------------------------------
    # Clean data
    # -------------------------------------------------
    df["title"] = df["title"].astype(str).str.strip()
    df["city"] = df["city"].astype(str).str.strip()

    df["prices"] = pd.to_numeric(df["prices"], errors="coerce").fillna(0)
    df[score_col] = pd.to_numeric(df[score_col], errors="coerce").fillna(0)

    # -------------------------------------------------
    # Build categories
    # -------------------------------------------------
    category_cols = [col for col in df.columns if "categories" in col.lower()]

    if category_cols:
        df["all_categories"] = df[category_cols].values.tolist()
        df["all_categories"] = df["all_categories"].apply(
            lambda x: [str(v).strip() for v in x if pd.notna(v) and str(v).strip() != ""]
        )
    elif "all_categories" in df.columns:
        df["all_categories"] = df["all_categories"].apply(
            lambda x: x if isinstance(x, list) else [str(x).strip()] if pd.notna(x) else []
        )
    else:
        df["all_categories"] = [[] for _ in range(len(df))]

    categories_exploded = df[["all_categories"]].explode("all_categories")
    if categories_exploded["all_categories"].notna().any():
        one_hot = pd.get_dummies(categories_exploded["all_categories"])
        one_hot_encoded = one_hot.groupby(categories_exploded.index).sum()
        one_hot_encoded = one_hot_encoded.reindex(df.index, fill_value=0)
    else:
        one_hot_encoded = pd.DataFrame(index=df.index)

    needed_categories = [
        "Shopping mall",
        "Buddhist temple",
        "Shinto shrine",
        "Information centre",
        "Amusement park",
        "Amusement center",
        "Theme park",
    ]
    for cat in needed_categories:
        if cat not in one_hot_encoded.columns:
            one_hot_encoded[cat] = 0

    # -------------------------------------------------
    # Sets & parameters
    # -------------------------------------------------
    n = len(df)
    I = range(n)
    categories = list(one_hot_encoded.columns)

    title_to_idx = {df.loc[i, "title"]: i for i in I}

    r = df[score_col].to_dict()
    e = df["prices"].to_dict()

    # Budget logic
    avg_price = float(df["prices"].mean()) if n > 0 else 0.0
    E = budget if budget is not None else avg_price * 9

    # g_i
    park_cats = ["Amusement park", "Amusement center", "Theme park"]
    g = {
        i: int(any(one_hot_encoded.loc[i, cat] == 1 for cat in park_cats))
        for i in I
    }

    # -------------------------------------------------
    # Model
    # -------------------------------------------------
    model = gp.Model("attraction_finding")
    model.setParam("OutputFlag", 0)

    y = model.addVars(I, vtype=GRB.BINARY, name="y")
    z = model.addVars(categories, vtype=GRB.BINARY, name="z")

    # Objective
    model.setObjective(quicksum(r[i] * y[i] for i in I), GRB.MAXIMIZE)

    # -------------------------------------------------
    # Constraints
    # -------------------------------------------------
    model.addConstr(quicksum(y[i] for i in I) == 9)

    model.addConstr(
        quicksum(one_hot_encoded.loc[i, "Shopping mall"] * y[i] for i in I) <= 2
    )

    model.addConstr(
        z["Buddhist temple"] + z["Shinto shrine"] <= 1
    )

    model.addConstr(
        quicksum(one_hot_encoded.loc[i, "Information centre"] * y[i] for i in I) <= 0
    )

    shinjuku_idx = [i for i in I if "shinjuku" in df.loc[i, "city"].lower()]
    model.addConstr(quicksum(y[i] for i in shinjuku_idx) <= 3)

    model.addConstr(quicksum(g[i] * y[i] for i in I) >= 1)

    # Link z_j
    for cat in categories:
        model.addConstr(quicksum(one_hot_encoded.loc[i, cat] * y[i] for i in I) >= z[cat])
        model.addConstr(quicksum(one_hot_encoded.loc[i, cat] * y[i] for i in I) <= n * z[cat])

    # Special constraints
    if "Tokyo Dome City" in title_to_idx and "Shinjuku Gyoen Cherry Tree Area" in title_to_idx:
        model.addConstr(
            y[title_to_idx["Tokyo Dome City"]] <= y[title_to_idx["Shinjuku Gyoen Cherry Tree Area"]]
        )

    if "Tokyo Camii & Diyanet Turkish Culture Center" in title_to_idx:
        model.addConstr(
            y[title_to_idx["Tokyo Camii & Diyanet Turkish Culture Center"]] == 0
        )

    special_titles = [
        "Ninja Trick House In Tokyo",
        "Samurai Ninja Museum Asakusa Tokyo",
        "Asakusa Hanayashiki"
    ]
    special_idx = [title_to_idx[t] for t in special_titles if t in title_to_idx]
    if special_idx:
        model.addConstr(quicksum(y[i] for i in special_idx) <= 2)

    # Budget constraint
    model.addConstr(quicksum(e[i] * y[i] for i in I) <= E)

    # -------------------------------------------------
    # Optimize
    # -------------------------------------------------
    model.optimize()

    # -------------------------------------------------
    # Results
    # -------------------------------------------------
    if model.status == GRB.OPTIMAL:
        selected_idx = [i for i in I if y[i].X > 0.5]
        selected_df = df.loc[selected_idx].copy()

        total_price = sum(e[i] for i in selected_idx)

        print("\n✅ Optimal solution found")
        print(f"Objective value (total review score): {model.ObjVal:.4f}")
        print(f"Total price: {total_price:.2f}")
        print(f"Budget used: {E:.2f}")
        print("\n📍 Selected Attractions:")

        for idx in selected_idx:
            print(f"- {df.loc[idx, 'title']} (City: {df.loc[idx, 'city']}, Score: {r[idx]:.2f}, Price: {e[idx]:.2f})")

        return selected_df

    else:
        print(f"No optimal solution found. Status: {model.status}")
        return pd.DataFrame()

In [17]:
selected_attractions = solve_attraction_finding_problem(attractions, budget=12400.79)


✅ Optimal solution found
Objective value (total review score): 41.5000
Total price: 9100.00
Budget used: 12400.79

📍 Selected Attractions:
- Shinjuku Gyoen National Garden (City: Shinjuku, Score: 4.60, Price: 500.00)
- Meiji Jingu (City: Shibuya, Score: 4.60, Price: 0.00)
- Statue of Liberty (City: Minato, Score: 4.50, Price: 0.00)
- Tokyo Station Marunouchi South Gate Dome (City: Chiyoda, Score: 4.60, Price: 0.00)
- Shinjuku Gyoen Cherry Tree Area (City: Shibuya, Score: 4.70, Price: 500.00)
- Marunouchi Tokyo Station Square (City: Chiyoda, Score: 4.50, Price: 0.00)
- Unicorn Gundam (City: Koto, Score: 4.60, Price: 0.00)
- teamLab Planets TOKYO DMM (City: Koto, Score: 4.50, Price: 4800.00)
- Samurai Ninja Museum Asakusa Tokyo (City: Taito, Score: 4.90, Price: 3300.00)


In [18]:
selected_attractions.to_csv("selected_attractions.csv")

## What I Already Have

I already have:

- Optimal attractions and their latitude and longitude in selected_df["title", "latitude", "longitude"]
- A folium map named [m]

## Response Rules for GenAI

Please strictly follow these rules:

1. Do NOT reimport any libraries that have been imported before.
2. Output **Python code only**.

# Task for GenAI

- Provide code to plot the attractions on the folium map [m_optimised]

In [15]:
m_optimised = folium.Map(
location=[selected_attractions["latitude"].mean(), selected_attractions["longitude"].mean()],
zoom_start=12
)

for _, row in selected_attractions[["title", "latitude", "longitude"]].dropna().iterrows():
    folium.Marker(
    location=[row["latitude"], row["longitude"]],
    popup=row["title"]
    ).add_to(m_optimised)

m_optimised

# Plan 2: Using weights to decide the attractions to go to

Assume that I want to create app that helps to tourist to pick attractions to go to by their requirements. They want to go to exactly 9 attractions in Tokyo. Suppose that I have n attractions and m categories. Here are my criterias:

Here is my reference setting:

| Preference           | Review Weight (w_r) | Entrance Fee Weight (w_e) |
|---------------------|--------------------|---------------------------|
| Premium Experience  | 0.8                | 0.2                       |
| Balanced Experience | 0.5                | 0.5                       |
| Budget-friendly     | 0.2                | 0.8                       |

Variables:
- y_i: Binary variable where attraction i is selected (Decision Variable)
- r_i: Review score of attraction i 
- e_i: Entrance fee of each attractions, column name "prices" 
- A : A set of attraction which is a subset of {1,...,n}
- C : A set of categories which is a subset of {1,...,m}
- a_{ij}: Binary variable if attraction i is truely category j 
- z_j: A binary variable where z_j = 1 if category j is represented at least once, otherwise z_j = 0
- g_i: Binary variable where g_i = 1 if attraction i belongs to at least one category in group C

In this case, we try using the premium experience

Objective function: max sum{i = 1 to n}sum{i = 1 to m}((w_r* r_i^norm - w_e* e_i^norm)* x_{ij})

s.t
- sum{i = 1 to n} y_i = 9
- sum{i = 1 to n} a_{i, "Shopping mall"} * yi <= 2
- z_{"Buddhist temple"} + z_{"Shinto shrine"} <= 1
- sum{i = 1 to n} a_{i, "Information centre"} * yi <= 0
- sum{i is an element of a set that contains "Shinjuku" under the "city" column in attractions } y_i <= 3
- sum{i = 1 to n} g_i * y_i >=  1 where g_i = 1 if attraction i belongs to at least "Amusement park", "Asumement center" or "Theme park"
- y_{"Tokyo Dome City} <= y_{"Shinjuku Gyoen Cherry Tree Area"}
- y{"Tokyo Camii & Diyanet Turkish Culture Center"} = 0 
- y_{"Ninja Trick House In Tokyo"} + y_{"Samurai Ninja Museum Asakusa Tokyo"} + y_{"Asakusa Hanayashiki"} <=2
- y_i is binary

I want to see two columns: r_i^norm and e_i^norm in the df
- r_i^norm = (r_i - r_min)/(r_max - r_min)
- e_i^norm = (e_i - e_min)/(e_max - e_min)

Can you provide Python code using Gurobi to solve the problem? Please make the solver a function named "solve_attraction_finding_weighted". 

In [32]:
def solve_attraction_finding_weighted(attractions, n_select=9):
    import pandas as pd
    import gurobipy as gp
    from gurobipy import GRB, quicksum

    df = attractions.copy().reset_index(drop=True)

    # -------------------------------------------------
    # Validation
    # -------------------------------------------------
    required_cols = ["title", "city", "prices"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required column(s): {missing_cols}")

    if n_select <= 0 or n_select > len(df):
        raise ValueError("n_select must be between 1 and number of attractions")

    # -------------------------------------------------
    # Review score column
    # -------------------------------------------------
    if "review_score" in df.columns:
        score_col = "review_score"
    elif "totalScore" in df.columns:
        score_col = "totalScore"
    elif "r_i" in df.columns:
        score_col = "r_i"
    else:
        raise ValueError("No review score column found.")

    # -------------------------------------------------
    # Clean data
    # -------------------------------------------------
    df["title"] = df["title"].astype(str).str.strip()
    df["city"] = df["city"].astype(str).str.strip()
    df["prices"] = pd.to_numeric(df["prices"], errors="coerce").fillna(0)
    df[score_col] = pd.to_numeric(df[score_col], errors="coerce").fillna(0)

    # -------------------------------------------------
    # Categories
    # -------------------------------------------------
    category_cols = [col for col in df.columns if "categories" in col.lower()]

    if category_cols:
        df["all_categories"] = df[category_cols].values.tolist()
        df["all_categories"] = df["all_categories"].apply(
            lambda x: [str(v).strip() for v in x if pd.notna(v) and str(v).strip() != ""]
        )
    else:
        df["all_categories"] = [[] for _ in range(len(df))]

    categories_exploded = df[["all_categories"]].explode("all_categories")

    if categories_exploded["all_categories"].notna().any():
        one_hot = pd.get_dummies(categories_exploded["all_categories"])
        one_hot_encoded = one_hot.groupby(categories_exploded.index).sum()
        one_hot_encoded = one_hot_encoded.reindex(df.index, fill_value=0)
    else:
        one_hot_encoded = pd.DataFrame(index=df.index)

    needed_categories = [
        "Shopping mall",
        "Buddhist temple",
        "Shinto shrine",
        "Information centre",
        "Amusement park",
        "Amusement center",
        "Theme park",
    ]
    for cat in needed_categories:
        if cat not in one_hot_encoded.columns:
            one_hot_encoded[cat] = 0

    # -------------------------------------------------
    # Normalization
    # -------------------------------------------------
    r_min, r_max = df[score_col].min(), df[score_col].max()
    e_min, e_max = df["prices"].min(), df["prices"].max()

    df["r_i_norm"] = (df[score_col] - r_min) / (r_max - r_min) if r_max > r_min else 0
    df["e_i_norm"] = (df["prices"] - e_min) / (e_max - e_min) if e_max > e_min else 0

    # -------------------------------------------------
    # Parameters
    # -------------------------------------------------
    w_r, w_e = 0.8, 0.2

    n = len(df)
    I = range(n)
    title_to_idx = {df.loc[i, "title"]: i for i in I}

    r_norm = df["r_i_norm"].to_dict()
    e_norm = df["e_i_norm"].to_dict()

    group_cats = ["Amusement park", "Amusement center", "Theme park"]
    g = {
        i: int(any(one_hot_encoded.loc[i, cat] == 1 for cat in group_cats))
        for i in I
    }

    # -------------------------------------------------
    # Model
    # -------------------------------------------------
    model = gp.Model("attraction_finding")
    model.setParam("OutputFlag", 0)

    y = model.addVars(I, vtype=GRB.BINARY)

    model.setObjective(
        quicksum((w_r * r_norm[i] - w_e * e_norm[i]) * y[i] for i in I),
        GRB.MAXIMIZE
    )

    # -------------------------------------------------
    # Constraints
    # -------------------------------------------------
    model.addConstr(quicksum(y[i] for i in I) == n_select)

    model.addConstr(
        quicksum(one_hot_encoded.loc[i, "Shopping mall"] * y[i] for i in I) <= 2
    )

    model.addConstr(
        quicksum(one_hot_encoded.loc[i, "Information centre"] * y[i] for i in I) <= 0
    )

    shinjuku_idx = [i for i in I if "shinjuku" in df.loc[i, "city"].lower()]
    model.addConstr(quicksum(y[i] for i in shinjuku_idx) <= 3)

    model.addConstr(quicksum(g[i] * y[i] for i in I) >= 1)

    if "Tokyo Dome City" in title_to_idx and "Shinjuku Gyoen Cherry Tree Area" in title_to_idx:
        model.addConstr(
            y[title_to_idx["Tokyo Dome City"]] <= y[title_to_idx["Shinjuku Gyoen Cherry Tree Area"]]
        )

    if "Tokyo Camii & Diyanet Turkish Culture Center" in title_to_idx:
        model.addConstr(
            y[title_to_idx["Tokyo Camii & Diyanet Turkish Culture Center"]] == 0
        )

    special_titles = [
        "Ninja Trick House In Tokyo",
        "Samurai Ninja Museum Asakusa Tokyo",
        "Asakusa Hanayashiki"
    ]
    special_idx = [title_to_idx[t] for t in special_titles if t in title_to_idx]
    if special_idx:
        model.addConstr(quicksum(y[i] for i in special_idx) <= 2)

    # -------------------------------------------------
    # Solve
    # -------------------------------------------------
    model.optimize()

    # -------------------------------------------------
    # Output
    # -------------------------------------------------
    if model.status == GRB.OPTIMAL:
        selected_idx = [i for i in I if y[i].X > 0.5]
        selected_df = df.loc[selected_idx].copy()

        selected_df["premium_utility"] = (
            w_r * selected_df["r_i_norm"] - w_e * selected_df["e_i_norm"]
        )

        print("\n✅ Optimal solution found")
        print(f"Number of attractions selected: {len(selected_df)}")
        print(f"Objective value: {model.ObjVal:.6f}")
        print("\n📍 Selected Attractions:")
        print(
            selected_df[
                ["title", "city", score_col, "prices", "r_i_norm", "e_i_norm", "premium_utility"]
            ].to_string(index=False)
        )

        return selected_df.reset_index(drop=True)

    else:
        print("❌ No optimal solution found")
        return pd.DataFrame()

In [33]:
selected_df = solve_attraction_finding_weighted(attractions)


✅ Optimal solution found
Number of attractions selected: 9
Objective value: 5.420833

📍 Selected Attractions:
                                     title     city  totalScore  prices  r_i_norm  e_i_norm  premium_utility
            Shinjuku Gyoen National Garden Shinjuku         4.6     500  0.785714  0.104167         0.607738
                               Meiji Jingu  Shibuya         4.6       0  0.785714  0.000000         0.628571
                           Tokyo Dome City   Bunkyo         4.3       0  0.571429  0.000000         0.457143
                                Cat Street  Shibuya         4.5       0  0.714286  0.000000         0.571429
  Tokyo Station Marunouchi South Gate Dome  Chiyoda         4.6       0  0.785714  0.000000         0.628571
           Shinjuku Gyoen Cherry Tree Area  Shibuya         4.7     500  0.857143  0.104167         0.664881
                            Unicorn Gundam     Koto         4.6       0  0.785714  0.000000         0.628571
        Samurai N

## What I Already Have

I already have:

- Optimal attractions and their latitude and longitude in selected_df["title", "latitude", "longitude"]
- A folium map named [m]

## Response Rules for GenAI

Please strictly follow these rules:

1. Do NOT reimport any libraries that have been imported before.
2. Output **Python code only**.

# Task for GenAI

- Provide code to plot the attractions on the folium map [m_optimised]

In [34]:
m_optimised = folium.Map(
location=[selected_df["latitude"].mean(), selected_df["longitude"].mean()],
zoom_start=12
)

for _, row in selected_df[["title", "latitude", "longitude"]].dropna().iterrows():
    folium.Marker(
    location=[row["latitude"], row["longitude"]],
    popup=row["title"]
    ).add_to(m_optimised)

m_optimised

In [ ]:
## Put the link(s) to your chatlog here:
https://chatgpt.com/share/69d6793a-bb9c-839c-8f64-175d97a5ca66